Title: uncertainties_corrections.ipynb

Author: Zachary Lane, Quin Aicken Davies

Date: 07/11/2025

Description: Perform corrections and calculate uncertainties

In [ ]:
import numpy as np
import requests
from io import StringIO

In [ ]:
def mag_error(f, f_error, zp_error):
    delta_f_term = 2.5/np.log(10) * (f_error/f)
    delta_zp_term = zp_error

    return np.sqrt(delta_f_term**2 + delta_zp_term**2)

def mag_to_jansky(mag):
    return 10**((8.9 - mag)/2.5)

def jansky_error(mag, mag_error):
    return np.sqrt((-0.4 * np.log(10) * 10**((8.9 - mag)/2.5))**2 * mag_error**2)

def zp_calculation(data, wcs, hdr, mjd, master_image_name, file):
    filtered_positions = filtering_positions(data, pix_tol=15, dist_tol = 20)
    print(f"Number of sources: {len(filtered_positions)}")

    skymapper_df_trunc, indexes_smap = skymapper_comp(filtered_positions, wcs, sr = 3/3600)
    print(f"Number of matched sources: {len(skymapper_df_trunc)}")
    return skymapper_df_trunc
    
def fetch_data(url):
    try:
        response = requests.get(url)
        # Check if the request was successful (status code 200)
        if response.status_code == 200:
            return response.text
        else:
            print(f"Failed to retrieve data. Status code: {response.status_code}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data: {e}")
        return None

def remove_duplicates(positions, tolerance=15):
    unique_positions = []
    for pos in positions:
        is_unique = True
        for unique_pos in unique_positions:
            if np.abs(pos[0] - unique_pos[0]) <= tolerance and np.abs(pos[1] - unique_pos[1]) <= tolerance:
                is_unique = False
                break
        if is_unique:
            unique_positions.append(pos)
    return np.array(unique_positions)

def filtering_positions(input_data, pix_tol=25, dist_tol = 15):
    temp_data = deepcopy(input_data)
    mean, median, std = sigma_clipped_stats(temp_data, sigma=5.0)  # Adjust sigma as needed

    # Initialize DAOStarFinder with the background statistics
    daofind = DAOStarFinder(fwhm=3.0, threshold=17.0*std)  # Adjust fwhm and threshold as needed
    sources = daofind(temp_data) # Find sources in the image

    positions = np.zeros((len(sources), 2))
    positions[:,0] = sources['xcentroid'].value
    positions[:,1] = sources['ycentroid'].value

    condition = np.logical_and(np.logical_and(positions[:, 0] >= pix_tol, positions[:, 0] <= (2048 - pix_tol)),
                            np.logical_and(positions[:, 1] >= pix_tol, positions[:, 1] <= (2048 - pix_tol)))

    filtered_positions = positions[condition]

    filtered_positions = remove_duplicates(filtered_positions, dist_tol)

    return filtered_positions

def skymapper_comp(filtered_positions, wcs, sr = 5/3600):

    sr = str(sr)

    ra_temps, dec_temps = wcs.all_pix2world(filtered_positions[:,0], filtered_positions[:,1], 0)

    skymapper_df = pd.DataFrame(columns = ['RA', 'DEC', 'g', 'd_g', 'r', 'd_r', 'i', 'd_i'])

    indexes = []

    for i in range(len(ra_temps)):

        ra_temp = str(ra_temps[i])
        dec_temp = str(dec_temps[i])
        url = f"https://skymapper.anu.edu.au/sm-cone/public/query?RA={ra_temp}&DEC={dec_temp}&SR={sr}&RESPONSEFORMAT=CSV"
        url_data = fetch_data(url)

        if url_data:

            csv_data = StringIO(url_data)
            df = pd.read_csv(csv_data)

            try:
                temp_df = [[df['raj2000'][0], df['dej2000'][0], df['g_psf'][0], df['e_g_psf'][0], df['r_psf'][0], df['e_r_psf'][0], df['i_psf'][0], df['e_i_psf'][0]]]
            except:
                continue

            skymapper_df = pd.concat([skymapper_df, pd.DataFrame(temp_df, columns = ['RA', 'DEC', 'g', 'd_g', 'r', 'd_r', 'i', 'd_i'])], ignore_index=True)
            indexes.append(i)

        else:
            continue
    return skymapper_df, indexes
#Colour cuts to put in
g-r < 0.5
g-r > -0.2